Panic Attack Frequency Detection using Machine Learning

In [ ]:
# Install dependencies
!pip install pandas numpy scikit-learn seaborn matplotlib imbalanced-learn xgboost

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from imblearn.over_sampling import SMOTE

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df.drop(columns=["Timestamp","comments"], inplace=True, errors="ignore")

In [ ]:
df = df.dropna()

In [ ]:
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = le.fit_transform(df[col])

In [ ]:
df["Stress_Index"] = df["work_interfere"] + df["mental_health_consequence"]

df["Support_Index"] = df["coworkers"] + df["supervisor"]

df["Health_Risk"] = df["phys_health_consequence"] + df["mental_health_consequence"]

In [ ]:
X = df.drop("treatment", axis=1)

y = df["treatment"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
smote = SMOTE()

X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
lr = LogisticRegression(max_iter=2000)

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, pred_lr))

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, pred_rf))

In [ ]:
svm = SVC(kernel="rbf", probability=True)

svm.fit(X_train, y_train)

pred_svm = svm.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, pred_svm))

In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, pred_xgb))

In [ ]:
results = pd.DataFrame({
    "Model":["Logistic Regression","Random Forest","SVM","XGBoost"],
    "Accuracy":[
        accuracy_score(y_test,pred_lr),
        accuracy_score(y_test,pred_rf),
        accuracy_score(y_test,pred_svm),
        accuracy_score(y_test,pred_xgb)
    ]
})

results

In [ ]:
sns.barplot(data=results, x="Model", y="Accuracy")

plt.title("Model Accuracy Comparison")

plt.show()

In [ ]:
scores = cross_val_score(xgb, X, y, cv=10)

print("Cross Validation Accuracy:", scores.mean())

In [ ]:
params = {
    "n_estimators":[200,300,400],
    "max_depth":[4,6,8],
    "learning_rate":[0.01,0.05,0.1]
}

grid = GridSearchCV(
    XGBClassifier(),
    params,
    cv=5
)

grid.fit(X_train,y_train)

print("Best Parameters:", grid.best_params_)

In [ ]:
best_model = grid.best_estimator_

best_model.fit(X_train,y_train)

pred = best_model.predict(X_test)

print("Final Accuracy:", accuracy_score(y_test,pred))

In [ ]:
cm = confusion_matrix(y_test,pred)

sns.heatmap(cm, annot=True, fmt="d")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
importances = best_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature":X.columns,
    "Importance":importances
})

importance_df = importance_df.sort_values(by="Importance", ascending=False)

importance_df.head(10)

In [ ]:
sns.barplot(
    data=importance_df.head(10),
    x="Importance",
    y="Feature"
)

plt.title("Top Influential Features")

plt.show()

In [ ]:
import joblib

joblib.dump(best_model,"model.pkl")

joblib.dump(list(X.columns),"features.pkl")

In [ ]:
from google.colab import files

files.download("model.pkl")

files.download("features.pkl")